# 7.1 K-평균 군집화 (KMeans) - 주석 설명판

원본 `7_1.ipynb`의 코드에 설명 주석을 단 버전이에요.

**군집화란?**
- 분류/회귀(지도학습): 정답(y)이 있음 → `fit(X, y)`
- 군집화(비지도학습): 정답 없이 **비슷한 데이터끼리 묶음** → `fit(X)`

**KMeans 동작 방식**
1. 중심점 K개를 놓는다
2. 각 데이터를 가장 가까운 중심점의 그룹에 넣는다
3. 그룹에 속한 데이터들의 **평균 위치**로 중심점을 옮긴다
4. 중심점이 더 이상 안 움직일 때까지 2~3 반복

**전체 흐름**
1. 붓꽃(iris) 데이터를 KMeans로 3개 군집으로 나누기 → 실제 품종과 비교
2. PCA로 2차원으로 줄여서 군집 결과 시각화
3. make_blobs로 만든 가상 데이터로 군집화 + 중심점 시각화

## 1. 붓꽃 데이터 불러오기

In [ ]:
# 데이터에 정답(라벨)이 있으면 지도학습(분류), 없으면 비지도학습(군집화)
# → 여기서는 붓꽃 품종(정답)을 일부러 안 쓰고, 특징만으로 묶을 수 있는지 확인

from sklearn.datasets import load_iris
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

iris = load_iris()
# 피처 4개만 DataFrame으로 (정답인 iris.target은 아직 안 넣음)
df = pd.DataFrame(data=iris.data,
                  columns=['sepal_length', 'sepal_width', 'petal_length', 'petal_width'])
df.head()

## 2. KMeans로 3개 군집 만들기

In [ ]:
kmeans = KMeans(n_clusters=3,        # 몇 개의 그룹으로 나눌지 (K)
                init='k-means++',    # 처음 중심점을 서로 멀리 떨어지게 똑똑하게 고르는 방법 (기본값)
                max_iter=300,        # 중심점 이동을 최대 몇 번 반복할지
                random_state=0)      # 결과를 매번 똑같이 나오게 고정
kmeans.fit(df)    # ← 정답 없이 X만 넣음! (비지도학습)

# labels_: 각 데이터가 몇 번 군집에 속했는지 (0, 1, 2)
print(kmeans.labels_)
# ※ 군집 번호 0, 1, 2는 그냥 이름표일 뿐 품종 번호와 같은 뜻이 아님

## 3. 군집 결과와 실제 품종 비교

In [ ]:
df['target'] = iris.target       # 실제 품종 (0: setosa, 1: versicolor, 2: virginica)
df['cluster'] = kmeans.labels_   # KMeans가 나눈 군집 번호

# (실제 품종, 군집 번호) 조합별로 개수 세기
res = df.groupby(['target', 'cluster'])['sepal_length'].count()
res

# 결과 해석:
#  target 0 (setosa)     → 50개 전부 군집 1  ✅ 완벽하게 분리
#  target 1 (versicolor) → 48개 군집 0, 2개 군집 2  → 거의 맞음
#  target 2 (virginica)  → 36개 군집 2, 14개 군집 0 → 많이 섞임
# → versicolor와 virginica는 특징이 비슷해서 정답 없이 나누기 어려움

## 4. PCA로 2차원으로 줄이기 (시각화용)

In [ ]:
from sklearn.decomposition import PCA

# 피처가 4개라 그대로는 그래프로 못 그림 → PCA로 2개 성분으로 압축
pca = PCA(n_components=2)
pca_transformed = pca.fit_transform(iris.data)

df['pca_x'] = pca_transformed[:, 0]   # 첫 번째 주성분 → x축
df['pca_y'] = pca_transformed[:, 1]   # 두 번째 주성분 → y축
df.head()

## 5. 군집 결과 시각화

In [ ]:
# 군집 번호별로 해당하는 행의 인덱스(위치) 뽑기
marker0_ind = df[df['cluster'] == 0].index
marker1_ind = df[df['cluster'] == 1].index
marker2_ind = df[df['cluster'] == 2].index

# 군집마다 다른 모양으로 점 찍기: o(동그라미), s(네모), ^(세모)
# df.loc[행, 열] → 해당 군집의 pca_x, pca_y 값
plt.scatter(x=df.loc[marker0_ind, 'pca_x'], y=df.loc[marker0_ind, 'pca_y'], marker='o')
plt.scatter(x=df.loc[marker1_ind, 'pca_x'], y=df.loc[marker1_ind, 'pca_y'], marker='s')
plt.scatter(x=df.loc[marker2_ind, 'pca_x'], y=df.loc[marker2_ind, 'pca_y'], marker='^')

plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.title('3 Cluster Visualization by 2 PCA Components')
plt.show()

# 해석: 네모(군집 1 = setosa)는 왼쪽에 뚝 떨어져 있음
#       동그라미와 세모는 오른쪽에 붙어 있어서 경계가 겹침 → 3번 셀 결과와 같은 이야기

## 6. 가상 데이터 만들기 (make_blobs)

붓꽃처럼 군집이 겹치는 데이터 말고, **군집이 잘 나뉘는 연습용 데이터**를 직접 만들어서 KMeans를 확인해 봐요.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs

# make_blobs: 군집 연습용 가상 데이터를 만들어 주는 함수
X, y = make_blobs(n_samples=200,     # 데이터 개수
                  n_features=2,      # 피처 개수 (2개 → 바로 그래프로 그릴 수 있음)
                  centers=3,         # 군집(덩어리) 개수
                  cluster_std=0.8,   # 군집의 표준편차 (클수록 퍼지고 서로 겹침)
                  random_state=0)

print(X.shape, y.shape)   # (200, 2) (200,)
# X: 점의 좌표 / y: 각 점이 원래 어느 덩어리에서 만들어졌는지 (정답)

In [ ]:
# np.unique(..., return_counts=True): 고유값과 각 값의 개수를 같이 돌려줌
unique, counts = np.unique(y, return_counts=True)
print(unique, counts)   # [0 1 2] [67 67 66] → 덩어리마다 약 67개씩

In [ ]:
import pandas as pd

df = pd.DataFrame(data=X, columns=['ftr1', 'ftr2'])
df['target'] = y   # 원래 덩어리 번호
df.head()

## 7. 원래 덩어리 시각화 (정답)

In [ ]:
target_list = np.unique(y)                    # [0, 1, 2]
markers = ['o', 's', '^', 'P', 'D', 'H', 'x']  # 점 모양 목록

for target in target_list:
    target_cluster = df[df['target'] == target]   # 해당 덩어리의 데이터만
    plt.scatter(x=target_cluster['ftr1'], y=target_cluster['ftr2'],
                edgecolor='k',               # 점 테두리 검은색
                marker=markers[target])      # 덩어리 번호에 맞는 모양

plt.show()   # for문이 끝난 뒤 한 번만 → 그래프 1개에 모두 그림

## 8. KMeans 군집화 + 중심점 시각화

In [ ]:
kmeans = KMeans(n_clusters=3, init='k-means++', max_iter=200, random_state=0)

# fit_predict: 학습(fit)과 군집 번호 반환(predict)을 한 번에
cluster_labels = kmeans.fit_predict(X)
df['kmeans_label'] = cluster_labels

centers = kmeans.cluster_centers_          # 각 군집 중심점의 좌표 (3행 x 2열)
unique_labels = np.unique(cluster_labels)  # [0, 1, 2]
markers = ['o', 's', '^', 'P', 'D', 'H', 'x']

for label in unique_labels:
    label_cluster = df[df['kmeans_label'] == label]
    center_x_y = centers[label]   # 이 군집의 중심점 [x, y]

    # ① 군집에 속한 데이터 점들
    plt.scatter(x=label_cluster['ftr1'], y=label_cluster['ftr2'],
                edgecolors='k', marker=markers[label])

    # ② 중심점: 크고 하얀 도형 (배경 역할)
    plt.scatter(x=center_x_y[0], y=center_x_y[1], s=200,   # s: 점 크기
                color='white', alpha=0.9, edgecolors='k',  # alpha: 투명도
                marker=markers[label])

    # ③ 중심점 위에 군집 번호를 글자로 표시
    #    marker='$0$' 처럼 $ 사이에 글자를 넣으면 그 글자가 점 모양이 됨
    #    '$%d$' % label → label이 0이면 '$0$', 1이면 '$1$'
    plt.scatter(x=center_x_y[0], y=center_x_y[1], s=70,
                color='k', edgecolors='k',
                marker='$%d$' % label)

plt.show()   # ⚠ for문 밖! (안에 있으면 그래프가 군집 수만큼 따로 나옴)

## 9. 군집 결과와 원래 덩어리 비교

In [ ]:
# 원래 덩어리(target)별로 KMeans가 몇 번 군집에 넣었는지 개수 세기
df.groupby('target')['kmeans_label'].value_counts()

# 결과 해석:
#  target 0 → 66개 군집 0, 1개 군집 1
#  target 1 → 67개 전부 군집 2
#  target 2 → 65개 군집 1, 1개 군집 2
# → 200개 중 2개만 다른 군집에 들어감 = 거의 완벽
#   (번호는 달라도 '같은 덩어리끼리 묶였는지'가 중요)
#   cluster_std를 크게 하면 덩어리가 겹쳐서 붓꽃처럼 더 많이 섞임

## 정리

| | 붓꽃 데이터 | make_blobs 데이터 |
|---|---|---|
| 군집 결과 | setosa는 완벽, 나머지 두 품종은 섞임 | 거의 완벽 (200개 중 2개만 다름) |
| 이유 | versicolor와 virginica의 특징이 비슷함 | 덩어리가 서로 잘 떨어져 있음 |

**기억할 점**
- KMeans는 **정답 없이** `fit(X)`만으로 군집을 나눔
- 군집 개수 **K는 사람이 정해야 함** → 7_2의 **실루엣 계수**로 적절한 K를 찾음
- 군집 번호(0, 1, 2)는 이름표일 뿐, 실제 정답 번호와 같은 뜻이 아님
- 피처가 많으면 **PCA**로 2차원으로 줄여서 시각화